In [ ]:

# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

In [ ]:

# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

In [ ]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

import importlib
import captioning
importlib.reload(captioning)
from captioning import run_captioning

In [ ]:
# =============================
# CONTROLLED VERBOSITY
# =============================

# Disable HF download progress bars BEFORE importing anything HF-related

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [ ]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import random
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging


# Silence transformers & HF logs (keep only errors)
transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [ ]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

In [ ]:
# control flags
RUN_CAPTIONING = True
RUN_TEXT_VARIATION = False
RUN_IMAGE_GENERATION = False
RUN_TRAINING = False

In [ ]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

In [ ]:
# extract labels
labels = dataset_train._labels
indices = np.arange(len(dataset_train))

# perform stratified split
train_small_idx, _ = train_test_split(
    indices,
    train_size=0.30,
    stratify=labels,
    random_state=42
)

SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

np.save(os.path.join(SPLIT_DIR, "train_small_indices.npy"), train_small_idx)

In [ ]:
# # run for ALL
# # create subset dataset from training set
dataset_train_small = Subset(dataset_train, train_small_idx)

In [ ]:
# run for 10
dataset_train_small_10 = Subset(dataset_train, train_small_idx[:10])

# Captioning

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16 # to reduce GPU memory usage
)

model.to(device)
model.eval()

In [ ]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

if RUN_CAPTIONING:
    captions_dict = run_captioning(
        # dataset_train_small=dataset_train_small, # FOR ALL
        dataset_train_small=dataset_train_small_10, # FOR 10
        model=model,
        processor=processor,
        device=device,
        output_path=CAPTION_PATH
    )

In [ ]:
import nbformat
import os

def hard_clean_notebook(path):
    nb = nbformat.read(path, as_version=4)

    if "widgets" in nb.metadata:
        del nb.metadata["widgets"]

    for cell in nb.cells:
        if "widgets" in cell.get("metadata", {}):
            del cell["metadata"]["widgets"]

    nbformat.write(nb, path)
    print(f"Cleaned: {os.path.basename(path)}")


# 🔥 Walk entire project and clean every notebook
for root, _, files in os.walk(PROJECT_ROOT):
    for file in files:
        if file.endswith(".ipynb"):
            hard_clean_notebook(os.path.join(root, file))

print("All notebooks in project HARD cleaned.")

# Text variation

## FLAN-T5-Large Model



In [ ]:
import text_variation_flan_large
importlib.reload(text_variation_flan_large)
from text_variation_flan_large import run_text_variation as run_flan_large

In [ ]:
MAX_ITEMS = 10

In [ ]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [ ]:
print("\n=== FLAN LARGE ===")
run_flan_large(
    caption_file=CAPTION_PATH,
    max_items=MAX_ITEMS
)

In [ ]:
from text_variation_flan_xl import run_text_variation as run_flan_xl

run_flan_xl(
    caption_file=CAPTION_PATH,
    max_items=10
)

In [ ]:
        
from text_variation_mistral import run_text_variation


DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAPTIONS_DIR = os.path.join(DATA_DIR, "captions")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")


os.makedirs(TEXT_VARIATIONS_DIR, exist_ok=True)

CAPTION_FILE = os.path.join(
    CAPTIONS_DIR,
    "captions_train_small_10.json"
)

TEXT_VARIATION_FILE = os.path.join(
    TEXT_VARIATIONS_DIR,
    "text_variations_train_small_10.json"
)

run_text_variation(
    caption_file = CAPTION_FILE,
    output_file= TEXT_VARIATION_FILE,
    max_items=10
)

